# Phase 4: Ensemble Recommendation System

This notebook focuses on Phase 4 of the recommendation system, where we develop and evaluate an ensemble model using previous phase outputs. We combine scores from previous phases, including Neural Collaborative Filtering (NCF), Content-Based Filtering (CBF), and Wide & Deep (WND) models. The ensemble utilizes Logistic Regression and incorporates new features such as user and game cold-start flags to improve recommendation performance.

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import os
import random
import numpy as np
import pandas as pd

from pathlib import Path
_BASE = Path.cwd()  # Jupyter sets cwd to notebook directory

FEAT_DIR   = str(_BASE.parent / "phase0_finalised" / "feature_engineered_data")
NCF_DIR    = str(_BASE.parent / "phase1_finalised" / "phase1_outputs")
CBF_DIR    = str(_BASE.parent / "phase2_finalised" / "phase2_2" / "phase2_2_outputs")
WD_DIR     = str(_BASE.parent / "phase3_finalised" / "without_extra_filter" / "phase3_outputs")
OUTPUT_DIR = str(_BASE / "phase4_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(FEAT_DIR)
print(NCF_DIR)
print(WD_DIR)
print(OUTPUT_DIR)



d:\NUS\2526S2\BT4222\BT4222_Group5_AY2526_S2\phase0_finalised\feature_engineered_data
d:\NUS\2526S2\BT4222\BT4222_Group5_AY2526_S2\phase1_finalised\phase1_outputs
d:\NUS\2526S2\BT4222\BT4222_Group5_AY2526_S2\phase3_finalised\without_extra_filter\phase3_outputs
d:\NUS\2526S2\BT4222\BT4222_Group5_AY2526_S2\phase4_finalised\phase4_outputs


# Load Data

In [3]:
import sys
print(sys.executable)  # Should show venv path
pd.read_parquet("../phase0_finalised/feature_engineered_data/train_with_features.parquet").shape


c:\Users\Admin\AppData\Local\Programs\Python\Python311\python.exe


(173832, 36)

In [4]:
%pip install pyarrow fastparquet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Features
train_with_features = pd.read_parquet(os.path.join(FEAT_DIR, "train_with_features.parquet"))
test_with_features = pd.read_parquet(os.path.join(FEAT_DIR, "test_with_features.parquet"))
negatives_with_features = pd.read_parquet(os.path.join(FEAT_DIR, "negatives_with_features.parquet"))

In [6]:
print("Train: ", train_with_features.shape)
print("Test: ", test_with_features.shape)
print("Neg: ", negatives_with_features.shape)

train_with_features[['recommendationid', 'appid', 'author_steamid', 'user_review_count_loo','game_review_count_loo']].head()

Train:  (173832, 36)
Test:  (51063, 36)
Neg:  (690537, 14)


,recommendationid,appid,author_steamid,user_review_count_loo,game_review_count_loo
0,50984579,773390,76561197960268765,1.0,6.0
1,114858657,1199760,76561197960268765,1.0,15.0
2,164311814,2284370,76561197960272177,1.0,9.0
3,166102114,1262760,76561197960272177,1.0,3.0
4,177493091,2461580,76561197960272191,5.0,10.0


In [7]:
# Phase 1 outputs
ncf_test_scores = pd.read_parquet(os.path.join(NCF_DIR, "ncf_test_scores.parquet"))
ncf_train_scores = pd.read_parquet(os.path.join(NCF_DIR, "ncf_train_scores.parquet"))
ncf_neg_scores = pd.read_parquet(os.path.join(NCF_DIR, "ncf_neg_scores.parquet"))

In [8]:
print("NCF Train: ", ncf_train_scores.shape)
print("NCF Test: ", ncf_test_scores.shape)
print("NCF Neg: ", ncf_neg_scores.shape)

ncf_train_scores.head()

NCF Train:  (173832, 6)
NCF Test:  (51063, 6)
NCF Neg:  (690537, 6)


,author_steamid,appid,user_idx,item_idx,voted_up,ncf_score
0,76561197960268765,773390,0,9398,False,7.517765e-08
1,76561197960268765,1199760,0,15259,True,9.865758e-01
2,76561197960272177,2284370,1,26082,True,9.711171e-01
3,76561197960272177,1262760,1,16028,True,9.649174e-01
4,76561197960272191,2461580,2,27348,False,1.848527e-08


In [9]:
# Phase 2 outputs
content_similarity_scores = pd.read_parquet(os.path.join(CBF_DIR, "content_similarity_scores.parquet"))

In [10]:
print("CBF: ", content_similarity_scores.shape)

content_similarity_scores.head()

CBF:  (329070, 4)


,author_steamid,appid,content_sim_score,rank
0,76561197960268765,1163930,0.898305,1
1,76561197960268765,876320,0.895732,2
2,76561197960268765,842430,0.883686,3
3,76561197960268765,1307460,0.882411,4
4,76561197960268765,789220,0.881376,5


In [11]:
# Phase 3 outputs
wd_train_scores = pd.read_parquet(os.path.join(WD_DIR, "wd_train_scores.parquet"))
wd_test_scores = pd.read_parquet(os.path.join(WD_DIR, "wd_test_scores.parquet"))
wd_neg_scores = pd.read_parquet(os.path.join(WD_DIR, "wd_neg_scores.parquet"))

In [12]:
print("WD Train: ", wd_train_scores.shape)
print("WD Test: ", wd_test_scores.shape)
print("WD Neg: ", wd_neg_scores.shape)

wd_train_scores.head()

WD Train:  (173832, 4)
WD Test:  (51063, 4)
WD Neg:  (690537, 4)


,author_steamid,appid,voted_up,wd_score
0,76561197960268765,773390,False,0.948765
1,76561197960268765,1199760,True,0.992470
2,76561197960272177,2284370,True,0.288937
3,76561197960272177,1262760,True,0.957878
4,76561197960272191,2461580,False,0.886024


In [13]:
# Performance of previous models

# Phase 1 (Hard coded from phase 1 notebook)
phase1_results = pd.DataFrame({
    'Metric': ['Best val', 'Test'],
    'HR@10': [0.8933, 0.9104],
    'NDCG@10': [0.6592, 0.7193]
})
display(phase1_results)

# Phase 2 (Hard coded from phase 2 notebook)
phase2_results = pd.DataFrame({
    'Metric': ['Test'],
    'HR@10': [0.0459],
    'NDCG@10': [0.0277]
})
display(phase2_results)

# Phase 3
phase3_results = pd.read_csv(os.path.join(WD_DIR, "phase3_results.csv"))
display(phase3_results)

,Metric,HR@10,NDCG@10
0,Best val,0.8933,0.6592
1,Test,0.9104,0.7193


,Metric,HR@10,NDCG@10
0,Test,0.0459,0.0277


,Model,AUC-ROC,AUC-PR,F1,HR@10,NDCG@10
0,Logistic Regression,0.623697,0.824922,0.864637,0.828953,0.474423
1,XGBoost,0.723425,0.896158,0.864371,0.994666,0.955287
2,Wide & Deep,0.660428,0.845743,0.860140,0.808499,0.448857


# Feature Engineering: User and Game Cold-Start Flags

To address cold-start scenarios and provide more context to our ensemble model, we will create two new features:
1.  **User Cold-Start Flag**: Set to 1 if a user has less than 5 reviews, indicating a new or infrequent user.
2.  **Game Cold-Start Flag**: Set to 1 if a game has less than 3 reviews, indicating a newly released or less popular game.

These flags will be added to our combined feature sets for the meta-models.

In [14]:
# Create cold-start flags for each dataframe
train_with_features['user_cold_start_flag'] = (train_with_features['user_review_count_loo'] < 5).astype(int)
train_with_features['game_cold_start_flag'] = (train_with_features['game_review_count_loo'] < 3).astype(int)

test_with_features['user_cold_start_flag'] = (test_with_features['user_review_count_loo'] < 5).astype(int)
test_with_features['game_cold_start_flag'] = (test_with_features['game_review_count_loo'] < 3).astype(int)

negatives_with_features['user_cold_start_flag'] = (negatives_with_features['user_review_count_loo'] < 5).astype(int)
negatives_with_features['game_cold_start_flag'] = (negatives_with_features['game_review_count_loo'] < 3).astype(int)

# Preparing Training Data for the Ensemble Model

To build a robust ensemble, we need to train our meta-model on a dedicated training set. We will combine `ncf_train_scores` and `wd_train_scores` to create this training set.

In [15]:
print("NCF Train Scores:", ncf_train_scores.shape)
print("WND Train Scores:", wd_train_scores.shape)

# Merge NCF and WND train scores
train_scores = pd.merge(
    ncf_train_scores,
    wd_train_scores[['author_steamid', 'appid', 'voted_up', 'wd_score']],
    on=['author_steamid', 'appid', 'voted_up'],
    how='inner'
)

# For content_sim_score, since it's a general content similarity and not specific to train/test splits,
# we can merge it into both train and test sets. Fill NaNs with 0.
train_scores = pd.merge(
    train_scores,
    content_similarity_scores[['author_steamid', 'appid', 'content_sim_score']],
    on=['author_steamid', 'appid'],
    how='left'
)

train_scores = pd.merge(
    train_scores,
    train_with_features[['recommendationid', 'appid', 'author_steamid', 'user_review_count_loo','game_review_count_loo']],
    on=['author_steamid', 'appid'], how='left'
)

train_scores['content_sim_score'] = train_scores['content_sim_score'].fillna(0)

print("Final Combined Train Scores after adding CBF scores:", train_scores.shape)
display(train_scores.head())

NCF Train Scores: (173832, 6)
WND Train Scores: (173832, 4)
Final Combined Train Scores after adding CBF scores: (173862, 11)


,author_steamid,appid,user_idx,item_idx,voted_up,ncf_score,wd_score,content_sim_score,recommendationid,user_review_count_loo,game_review_count_loo
0,76561197960268765,773390,0,9398,False,7.517765e-08,0.948765,0.0,50984579,1.0,6.0
1,76561197960268765,1199760,0,15259,True,9.865758e-01,0.992470,0.0,114858657,1.0,15.0
2,76561197960272177,2284370,1,26082,True,9.711171e-01,0.288937,0.0,164311814,1.0,9.0
3,76561197960272177,1262760,1,16028,True,9.649174e-01,0.957878,0.0,166102114,1.0,3.0
4,76561197960272191,2461580,2,27348,False,1.848527e-08,0.886024,0.0,177493091,5.0,10.0


In [16]:
print("NCF Test Scores:", ncf_test_scores.shape)
print("WND Test Scores:", wd_test_scores.shape)

# Merge NCF and WND test scores
test_scores = pd.merge(
    ncf_test_scores,
    wd_test_scores[['author_steamid', 'appid', 'voted_up', 'wd_score']],
    on=['author_steamid', 'appid', 'voted_up'],
    how='inner'
)

# For content_sim_score, since it's a general content similarity and not specific to train/test splits,
# we can merge it into both train and test sets. Fill NaNs with 0.
test_scores = pd.merge(
    test_scores,
    content_similarity_scores[['author_steamid', 'appid', 'content_sim_score']],
    on=['author_steamid', 'appid'],
    how='left'
)

test_scores = pd.merge(
    test_scores,
    test_with_features[['recommendationid', 'appid', 'author_steamid', 'user_review_count_loo','game_review_count_loo']],
    on=['author_steamid', 'appid'], how='left'
)

test_scores['content_sim_score'] = test_scores['content_sim_score'].fillna(0)

print("Final Combined Test Scores after adding CBF scores:", test_scores.shape)
display(test_scores.head())

NCF Test Scores: (51063, 6)
WND Test Scores: (51063, 4)
Final Combined Test Scores after adding CBF scores: (51063, 11)


,author_steamid,appid,user_idx,item_idx,voted_up,ncf_score,wd_score,content_sim_score,recommendationid,user_review_count_loo,game_review_count_loo
0,76561197960268765,3595270,0,32229,False,0.983929,0.314235,0.0,203361951,2.0,6.0
1,76561197960272177,2176510,1,25201,True,0.000102,0.396653,0.0,192222400,2.0,1.0
2,76561197960272191,1141760,2,14461,False,0.000023,0.156225,0.0,201333597,6.0,1.0
3,76561197960273880,2297960,3,26194,True,0.989124,0.303086,0.0,163448272,17.0,31.0
4,76561197960273880,2433710,3,27164,True,0.937896,0.321212,0.0,165346433,17.0,6.0


# Train Ensemble Meta-Models with Cold-Start Features

Now, we will train our Logistic Regression and MLP meta-models, incorporating the new `user_cold_start_flag` and `game_cold_start_flag` features. This will allow the models to learn from these contextual indicators and potentially improve their predictions, especially for cold-start cases.

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

# Define features and target for the meta-model training (with new features)
X_train = train_scores[[
    'ncf_score', 'wd_score', 'content_sim_score',
    'user_review_count_loo', 'game_review_count_loo'
]]
y_train = train_scores['voted_up']

# Initialize and train the Logistic Regression meta-model with cold-start features
ensemble_model_lr = LogisticRegression(solver='liblinear', random_state=42)
ensemble_model_lr.fit(X_train, y_train)

print("Logistic Regression Meta-model training complete.")

# Define features and target for the meta-model evaluation (using the prepared test set)
X_test = test_scores[[
    'ncf_score', 'wd_score', 'content_sim_score',
    'user_review_count_loo', 'game_review_count_loo'
]]
y_test = test_scores['voted_up']

# Predict probabilities on the test set using the trained meta-model
test_scores['ensemble_score_lr'] = ensemble_model_lr.predict_proba(X_test)[:, 1]

# Evaluate the ensemble model's performance on the test set
auc_ensemble_lr = roc_auc_score(y_test, test_scores['ensemble_score_lr'])
acc_ensemble_lr = accuracy_score(y_test, (test_scores['ensemble_score_lr'] > 0.5))

print(f"\nAUC on Test Set: {auc_ensemble_lr:.4f}")
print(f"Accuracy on Test Set: {acc_ensemble_lr:.4f}")

display(test_scores.head())

Logistic Regression Meta-model training complete.

AUC on Test Set: 0.6728
Accuracy on Test Set: 0.5107


,author_steamid,appid,user_idx,item_idx,voted_up,ncf_score,wd_score,content_sim_score,recommendationid,user_review_count_loo,game_review_count_loo,ensemble_score_lr
0,76561197960268765,3595270,0,32229,False,0.983929,0.314235,0.0,203361951,2.0,6.0,0.999650
1,76561197960272177,2176510,1,25201,True,0.000102,0.396653,0.0,192222400,2.0,1.0,0.060652
2,76561197960272191,1141760,2,14461,False,0.000023,0.156225,0.0,201333597,6.0,1.0,0.042868
3,76561197960273880,2297960,3,26194,True,0.989124,0.303086,0.0,163448272,17.0,31.0,0.999638
4,76561197960273880,2433710,3,27164,True,0.937896,0.321212,0.0,165346433,17.0,6.0,0.999415


In [18]:
# Model coefficients
coef_table = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": ensemble_model_lr.coef_[0].round(4)
})

# Add intercept
intercept = pd.DataFrame({
    "Feature": ["Intercept"],
    "Coefficient": [ensemble_model_lr.intercept_[0].round(4)]
})

coef_table = pd.concat([intercept, coef_table], ignore_index=True)
coef_table

,Feature,Coefficient
0,Intercept,-3.3311
1,ncf_score,11.0089
2,wd_score,1.4988
3,content_sim_score,-5.3770
4,user_review_count_loo,-0.0011
5,game_review_count_loo,-0.0023


### Interpretation of Logistic Regression Coefficients

The `coef_table` provides insights into how each feature influences the log-odds of a positive recommendation in our ensemble Logistic Regression model. A positive coefficient indicates that an increase in the feature's value increases the log-odds (and thus the probability) of a positive recommendation, while a negative coefficient indicates the opposite.

*   **Intercept**: `-7.6197`
    *   **Significance**: The intercept represents the log-odds of a positive recommendation when all other features are zero. A negative intercept of `-7.6197` suggests that without any positive influence from the individual model scores or user/game review counts, the baseline probability of a positive recommendation is very low.

*   **ncf_score**: `10.4563`
    *   **Direction**: Positive.
    *   **Magnitude**: This is the largest positive coefficient, indicating a strong positive influence.
    *   **Interpretation**: A higher NCF score *strongly increases* the likelihood of a positive recommendation. This suggests that the Neural Collaborative Filtering model's predictions are a highly significant and positive indicator for the ensemble.

*   **wd_score**: `5.7509`
    *   **Direction**: Positive.
    *   **Magnitude**: A significant positive coefficient, though smaller than `ncf_score`.
    *   **Interpretation**: A higher Wide & Deep score *significantly increases* the likelihood of a positive recommendation. The Wide & Deep model's predictions also contribute positively and substantially to the ensemble's decision.

*   **content_sim_score**: `-5.4770`
    *   **Direction**: Negative.
    *   **Magnitude**: A significant negative coefficient.
    *   **Interpretation**: Surprisingly, a higher content similarity score *decreases* the likelihood of a positive recommendation. This could imply that while content similarity might be relevant for generating candidates, in the context of this ensemble model, a high `content_sim_score` in combination with other features (especially NCF and WND) might signal a less preferred item for the final positive recommendation. It might also indicate that content-based recommendations alone are often not sufficient or even misleading when user preferences (captured by NCF/WND) are strong.

*   **user_review_count_loo**: `-0.0009`
    *   **Direction**: Negative.
    *   **Magnitude**: Very small negative coefficient.
    *   **Interpretation**: A higher number of user reviews (excluding the current one) *slightly decreases* the likelihood of a positive recommendation. Given the small magnitude, its practical impact is negligible. It might suggest a very slight bias against users who have reviewed many games, perhaps indicating more critical users, but this effect is minimal.

*   **game_review_count_loo**: `-0.0064`
    *   **Direction**: Negative.
    *   **Magnitude**: Very small negative coefficient.
    *   **Interpretation**: A higher number of game reviews (excluding the current one) *slightly decreases* the likelihood of a positive recommendation. Similar to `user_review_count_loo`, the magnitude is very small, implying a negligible practical impact. It could subtly indicate that very popular games (with many reviews) are slightly less likely to receive a new positive recommendation *from this model*, but the effect is too small to be meaningful.

# Model Evaluation

In [19]:
from sklearn.metrics import roc_auc_score, f1_score

# Function to calculate Hit Rate at K
def hit_rate_at_k(df, user_col, item_col, true_label_col, score_col, k):
    """
    Calculates Hit Rate @ K.
    For each user, if any of the top K recommended items are true positives, it's a hit.
    """
    # Group by user and get top K recommendations based on score
    top_k_recs = df.groupby(user_col).apply(
        lambda x: x.nlargest(k, score_col)
    ).reset_index(drop=True)

    # Check if any of the top K recommendations are true positives
    hits = top_k_recs[top_k_recs[true_label_col] == True]

    # Count unique users who have at least one hit
    num_hits = hits[user_col].nunique()
    num_users = df[user_col].nunique()

    return num_hits / num_users if num_users > 0 else 0

# Function to calculate NDCG at K
def ndcg_at_k(df, user_col, item_col, true_label_col, score_col, k):
    """
    Calculates Normalized Discounted Cumulative Gain @ K.
    """
    ndcg_list = []
    for user_id in df[user_col].unique():
        user_data = df[df[user_col] == user_id].copy() # Use .copy() to avoid SettingWithCopyWarning
        user_data_sorted = user_data.sort_values(by=score_col, ascending=False)
        user_data_top_k = user_data_sorted.head(k)

        dcg = 0
        for i, row in enumerate(user_data_top_k.itertuples()):
            relevance = 1 if getattr(row, true_label_col) == True else 0
            dcg += relevance / np.log2(i + 2)

        idcg = 0
        ideal_relevance_scores = user_data.sort_values(by=true_label_col, ascending=False).head(k)[true_label_col].apply(lambda x: 1 if x == True else 0).values
        for i, relevance in enumerate(ideal_relevance_scores):
            idcg += relevance / np.log2(i + 2)

        if idcg > 0:
            ndcg_list.append(dcg / idcg)
        else:
            ndcg_list.append(0)

    return np.mean(ndcg_list) if ndcg_list else 0

# Store results in a dictionary
results = {}

# Define models and their corresponding score columns
metrics_to_evaluate = {
    'Logistic Regression': 'ensemble_score_lr'
}

# --- Evaluate on Train Set ---
# Predict probabilities on the train set using the trained meta-model
X_train_eval = train_scores[[
    'ncf_score', 'wd_score', 'content_sim_score',
    'user_review_count_loo', 'game_review_count_loo'
]]
# The ensemble_model_lr was already trained in a previous cell
train_scores['ensemble_score_lr'] = ensemble_model_lr.predict_proba(X_train_eval)[:, 1]

for model_name, score_col in metrics_to_evaluate.items():
    print(f"\nCalculating metrics for Train Set: {model_name}")

    y_true_train = train_scores['voted_up']
    y_pred_scores_train = train_scores[score_col]

    # Classification Metrics
    y_true_numeric_train = y_true_train.astype(int)

    auc_train = roc_auc_score(y_true_numeric_train, y_pred_scores_train)
    y_pred_binary_train = (y_pred_scores_train > 0.5).astype(int)
    f1_train = f1_score(y_true_numeric_train, y_pred_binary_train)

    # Ranking Metrics (K=10)
    hr_10_train = hit_rate_at_k(train_scores, 'author_steamid', 'appid', 'voted_up', score_col, k=10)
    ndcg_10_train = ndcg_at_k(train_scores, 'author_steamid', 'appid', 'voted_up', score_col, k=10)

    results[f'{model_name} (Train)'] = {
        'AUC-ROC': auc_train,
        'F1-score': f1_train,
        'HR@10': hr_10_train,
        'NDCG@10': ndcg_10_train
    }

# --- Evaluate on Test Set ---
for model_name, score_col in metrics_to_evaluate.items():
    print(f"\nCalculating metrics for Test Set: {model_name}")

    y_true_test = test_scores['voted_up']
    y_pred_scores_test = test_scores[score_col]

    # Classification Metrics
    y_true_numeric_test = y_true_test.astype(int)

    auc_test = roc_auc_score(y_true_numeric_test, y_pred_scores_test)
    y_pred_binary_test = (y_pred_scores_test > 0.5).astype(int)
    f1_test = f1_score(y_true_numeric_test, y_pred_binary_test)

    # Ranking Metrics (K=10)
    hr_10_test = hit_rate_at_k(test_scores, 'author_steamid', 'appid', 'voted_up', score_col, k=10)
    ndcg_10_test = ndcg_at_k(test_scores, 'author_steamid', 'appid', 'voted_up', score_col, k=10)

    results[f'{model_name} (Test)'] = {
        'AUC-ROC': auc_test,
        'F1-score': f1_test,
        'HR@10': hr_10_test,
        'NDCG@10': ndcg_10_test
    }

# Convert results to a DataFrame for display
results_df = pd.DataFrame.from_dict(results, orient='index')

print("\nPerformance Comparison:")
display(results_df.round(4))



Calculating metrics for Train Set: Logistic Regression

Calculating metrics for Test Set: Logistic Regression

Performance Comparison:


,AUC-ROC,F1-score,HR@10,NDCG@10
Logistic Regression (Train),0.9916,0.9784,0.910,0.9067
Logistic Regression (Test),0.6728,0.5710,0.782,0.7754


In [20]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression
from IPython.display import display # Ensure display is available

# Function to calculate Hit Rate at K
def hit_rate_at_k(df, user_col, item_col, true_label_col, score_col, k):
    """
    Calculates Hit Rate @ K.
    For each user, if any of the top K recommended items are true positives, it's a hit.
    """
    # Group by user and get top K recommendations based on score
    top_k_recs = df.groupby(user_col).apply(
        lambda x: x.nlargest(k, score_col)
    ).reset_index(drop=True)

    # Check if any of the top K recommendations are true positives
    hits = top_k_recs[top_k_recs[true_label_col] == True]

    # Count unique users who have at least one hit
    num_hits = hits[user_col].nunique()
    num_users = df[user_col].nunique()

    return num_hits / num_users if num_users > 0 else 0

# Function to calculate NDCG at K
def ndcg_at_k(df, user_col, item_col, true_label_col, score_col, k):
    """
    Calculates Normalized Discounted Cumulative Gain @ K.
    """
    ndcg_list = []
    for user_id in df[user_col].unique():
        user_data = df[df[user_col] == user_id].copy() # Use .copy() to avoid SettingWithCopyWarning
        user_data_sorted = user_data.sort_values(by=score_col, ascending=False)
        user_data_top_k = user_data_sorted.head(k)

        dcg = 0
        for i, row in enumerate(user_data_top_k.itertuples()):
            relevance = 1 if getattr(row, true_label_col) == True else 0
            dcg += relevance / np.log2(i + 2)

        idcg = 0
        ideal_relevance_scores = user_data.sort_values(by=true_label_col, ascending=False).head(k)[true_label_col].apply(lambda x: 1 if x == True else 0).values
        for i, relevance in enumerate(ideal_relevance_scores):
            idcg += relevance / np.log2(i + 2)

        if idcg > 0:
            ndcg_list.append(dcg / idcg)
        else:
            ndcg_list.append(0)

    return np.mean(ndcg_list) if ndcg_list else 0

# Store results in a dictionary
results = {}

# Define models and their corresponding score columns
metrics_to_evaluate = {
    'Logistic Regression': 'ensemble_score_lr'
}

# --- Evaluate on Train Set ---
# Predict probabilities on the train set using the trained meta-model
X_train_eval = train_scores[[
    'ncf_score', 'wd_score', 'content_sim_score',
    'user_review_count_loo', 'game_review_count_loo'
]]
# The ensemble_model_lr was already trained in a previous cell
train_scores['ensemble_score_lr'] = ensemble_model_lr.predict_proba(X_train_eval)[:, 1]

for model_name, score_col in metrics_to_evaluate.items():
    print(f"\nCalculating metrics for Train Set: {model_name}")

    y_true_train = train_scores['voted_up']
    y_pred_scores_train = train_scores[score_col]

    # Classification Metrics
    y_true_numeric_train = y_true_train.astype(int)

    auc_train = roc_auc_score(y_true_numeric_train, y_pred_scores_train)
    y_pred_binary_train = (y_pred_scores_train > 0.5).astype(int)
    f1_train = f1_score(y_true_numeric_train, y_pred_binary_train)

    # Ranking Metrics (K=10)
    hr_10_train = hit_rate_at_k(train_scores, 'author_steamid', 'appid', 'voted_up', score_col, k=10)
    ndcg_10_train = ndcg_at_k(train_scores, 'author_steamid', 'appid', 'voted_up', score_col, k=10)

    results[f'{model_name} (Train)'] = {
        'AUC-ROC': auc_train,
        'F1-score': f1_train,
        'HR@10': hr_10_train,
        'NDCG@10': ndcg_10_train
    }

# --- Evaluate on Test Set ---
for model_name, score_col in metrics_to_evaluate.items():
    print(f"\nCalculating metrics for Test Set: {model_name}")

    y_true_test = test_scores['voted_up']
    y_pred_scores_test = test_scores[score_col]

    # Classification Metrics
    y_true_numeric_test = y_true_test.astype(int)

    auc_test = roc_auc_score(y_true_numeric_test, y_pred_scores_test)
    y_pred_binary_test = (y_pred_scores_test > 0.5).astype(int)
    f1_test = f1_score(y_true_numeric_test, y_pred_binary_test)

    # Ranking Metrics (K=10)
    hr_10_test = hit_rate_at_k(test_scores, 'author_steamid', 'appid', 'voted_up', score_col, k=10)
    ndcg_10_test = ndcg_at_k(test_scores, 'author_steamid', 'appid', 'voted_up', score_col, k=10)

    results[f'{model_name} (Test)'] = {
        'AUC-ROC': auc_test,
        'F1-score': f1_test,
        'HR@10': hr_10_test,
        'NDCG@10': ndcg_10_test
    }

# Convert results to a DataFrame for display
results_df = pd.DataFrame.from_dict(results, orient='index')

print("\nPerformance Comparison:")
display(results_df.round(4))


Calculating metrics for Train Set: Logistic Regression

Calculating metrics for Test Set: Logistic Regression

Performance Comparison:


,AUC-ROC,F1-score,HR@10,NDCG@10
Logistic Regression (Train),0.9916,0.9784,0.910,0.9067
Logistic Regression (Test),0.6728,0.5710,0.782,0.7754


In [21]:
# Prepare phase1_results
phase1_display = phase1_results[1:].round(4).rename(columns={'Metric': 'Model'})
phase1_display.loc[phase1_display['Model'] == 'Test', 'Model'] = 'NCF'

# Prepare phase2_results
phase2_display = phase2_results.round(4).rename(columns={'Metric': 'Model'})
phase2_display.loc[phase2_display['Model'] == 'Test', 'Model'] = 'CBF'

# Prepare phase3_results
phase3_display = phase3_results[2:].round(4).rename(columns={'F1': 'F1-score'})
phase3_display = phase3_display.drop(columns=['AUC-PR'])

# Prepare phase4_results
phase4_results = results_df.loc[['Logistic Regression (Test)']].round(4).rename(index={'Logistic Regression': 'Ensemble LR'}).reset_index().rename(columns={'index': 'Model'})

# Concatenate all results
all_results = pd.concat([phase1_display, phase2_display, phase3_display, phase4_results], ignore_index=True)

# Set 'Model' as index for better display
all_results = all_results.set_index('Model')

# Fill NaN values with an empty string as requested
all_results_blank_nan = all_results.fillna('')

print("\nPerformance Comparison Across All Phases:")
display(all_results_blank_nan)



Performance Comparison Across All Phases:


,HR@10,NDCG@10,AUC-ROC,F1-score
Model,,,,
NCF,0.9104,0.7193,,
CBF,0.0459,0.0277,,
Wide & Deep,0.8085,0.4489,0.6604,0.8601
Logistic Regression (Test),0.7820,0.7754,0.6728,0.571


# Save Output for Phase 4

In [22]:
# Save the test_scores dataframe with the ensemble scores
test_scores_output_path = os.path.join(OUTPUT_DIR, "lr_test_scores.parquet")
test_scores.to_parquet(test_scores_output_path, index=False)
print(f"Ensemble test scores saved to: {test_scores_output_path}")

# Save the train_scores dataframe with the ensemble scores
train_scores_output_path = os.path.join(OUTPUT_DIR, "lr_train_scores.parquet")
train_scores.to_parquet(train_scores_output_path, index=False)
print(f"Ensemble test scores saved to: {train_scores_output_path}")

# Save the performance comparison results
phase4_eval_path = os.path.join(OUTPUT_DIR, "phase4_eval.csv")
all_results_blank_nan.to_csv(phase4_eval_path, index=True)
print(f"Performance comparison results saved to: {phase4_eval_path}")

Ensemble test scores saved to: d:\NUS\2526S2\BT4222\BT4222_Group5_AY2526_S2\phase4_finalised\phase4_outputs\lr_test_scores.parquet
Ensemble test scores saved to: d:\NUS\2526S2\BT4222\BT4222_Group5_AY2526_S2\phase4_finalised\phase4_outputs\lr_train_scores.parquet
Performance comparison results saved to: d:\NUS\2526S2\BT4222\BT4222_Group5_AY2526_S2\phase4_finalised\phase4_outputs\phase4_eval.csv
